In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

In [2]:
%pwd

'c:\\Users\\NishantHP\\Desktop\\MLOps project\\research'

In [3]:
os.chdir('..')

In [4]:
from MLOps_project.utils.common import read_yaml,createDIr
from MLOps_project import logger

In [5]:
from pathlib import Path

class config_magr_tran:
    def __init__(self):
        self.config= read_yaml(Path('config/config.yaml'))
        self.param= read_yaml(Path('params.yaml'))
        self.shema= read_yaml(Path('schema.yaml'))
    
    def create_tranformation_dir(self):
        config=self.config.data_transformation
        createDIr([config.root_dir_trans])
        logger.info('trasformation dir created')
        return config

In [14]:
class FeatureEng:
    def __init__(self,config):
        self.config=config

    def featureEngineering(self):
        data=self.config.data_dir

        df=pd.read_csv(data)

        df['Churn']=df['Churn'].map({'Yes':0,'No':1})
        df['tenure_grp']=pd.cut(df['tenure'],
                        bins=[0,12,24,36,48,60,72],
                        labels=['0-1yrs','1-2yrs','2-3yrs','3-4yrs','4-5yrs','5-6yrs'])
        df['monthly_charge']=pd.cut(df['MonthlyCharges'],bins=3,labels=['low','medium','high'])
        df.drop('customerID',axis=1,inplace=True)
        services = ['PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']
        
        df['Total_services']=df[services].apply(lambda x:(x=='Yes').sum() ,axis=1) 
        logger.info(f'feature engineering done')
        df.to_csv(self.config.data_dir, index=False) 
        logger.info('data is now saved in csv')
        config=self.config
        return config  

In [12]:
class DataTransformaiton:
    def __init__(self,config):

        self.config= config

    def train_test(self):
        df=pd.read_csv(self.config.data_dir)
        train, test = train_test_split(df,test_size=0.2)
        train.to_csv(os.path.join(self.config.train_data_dir),index=False)
        test.to_csv(os.path.join(self.config.test_data_dir),index=False)

        logger.info("Splited data into training and test sets")
        logger.info(train.shape)
        logger.info(test.shape)

        print(train.shape)
        print(test.shape)
        

    

In [17]:
try:
    data_trans=config_magr_tran()
    config_tran=data_trans.create_tranformation_dir()
    feat_eng=FeatureEng(config_tran)
    feat_done=feat_eng.featureEngineering()
    tr_te_split=DataTransformaiton(feat_done)
    tr_te_split.train_test()
except Exception as e:
    raise e


[2026-04-16 15:53:46,836: INFO: common: yaml file<_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-16 15:53:46,838: INFO: common: yaml file<_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-16 15:53:46,842: INFO: common: yaml file<_io.TextIOWrapper name='schema.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-16 15:53:46,846: INFO: common: created directory at artifacts/data_transformation]
[2026-04-16 15:53:46,848: INFO: 946922550: trasformation dir created]
[2026-04-16 15:53:47,560: INFO: 61886532: feature engineering done]
[2026-04-16 15:53:47,597: INFO: 61886532: data is now saved in csv]
[2026-04-16 15:53:47,676: INFO: 1912435001: Splited data into training and test sets]
[2026-04-16 15:53:47,677: INFO: 1912435001: (5634, 23)]
[2026-04-16 15:53:47,678: INFO: 1912435001: (1409, 23)]
(5634, 23)
(1409, 23)
